# AstroDetect v0.2 · 00 · Pull ~15 GB of real labelled light curves from Hugging Face
**Settings:** Accelerator **None** · Internet **On** · (optional) add a Kaggle secret `HF_TOKEN` for faster, rate-limit-free downloads.
Then **Save Version → Save & Run All (Commit)**.

What changes from v0.1:
| | v0.1 | v0.2 (this) |
|---|---|---|
| Data | 20 % of stars, ~2 GB | **~15 GB** of raw lc.fits |
| Classes | 4, three of them simulated | **3, straight from the catalogue**: planet / candidate / false positive |
| Which stars | random | every star with a planet or FP signal is eligible, candidate-only stars capped at 1.5 × planets, filled to the byte budget |
| Sectors | 2 random (e.g. 42 + 92) | up to 3 **consecutive** |
| Stored as | raw FITS | compact per-star `.npz` (quality-masked, crowding-corrected, sorted): ~4× smaller |

**Output:** `stars_XXXX.tar` (100 stars each) + `jobs.json` + the 3 catalogue CSVs + `pull_progress.json`.
If the run stops early (time budget), attach this output as an input and commit again: finished tars are skipped.

In [ ]:
# CELL 0 - the AstroDetect v0.2 code, embedded (nothing to upload). Just run it.
ASTRODETECT_ZIP_B64 = '...'
import os, sys, io, base64, zipfile, shutil
CODE = '/kaggle/working/code' if os.path.isdir('/kaggle') else os.path.abspath('code')
shutil.rmtree(CODE, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(ASTRODETECT_ZIP_B64))).extractall(CODE)
sys.path.insert(0, CODE)
for m in [m for m in sys.modules if m.startswith('astrodetect')]:
    del sys.modules[m]
import astrodetect
print('astrodetect', astrodetect.__version__)

In [ ]:
!pip install -q huggingface_hub astropy
import json, glob, time, collections
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download
from astrodetect import build3 as B

# ---------------- settings ----------------
BUDGET_GB      = 15.0      # raw lc.fits to download
FILES_PER_STAR = 3         # consecutive sectors per star
CAND_RATIO     = 1.5       # candidate signals <= 1.5 x planet signals
THREADS        = 16
TIME_BUDGET_H  = 11.0
OUT            = '/kaggle/working'
# ------------------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    TOKEN = None
print('HF token:', 'yes' if TOKEN else 'no (anonymous - fine, a bit slower)')

## 1 · Catalogue: what labels exist

In [ ]:
get = lambda f: pd.read_csv(hf_hub_download(B.REPO, f, repo_type='dataset', token=TOKEN))
signals, lcs, targets = get('signals.csv'), get('lightcurves.csv'), get('targets.csv')
for name, df in (('signals', signals), ('lightcurves', lcs), ('targets', targets)):
    print(f'{name:12s} {len(df):7d} rows | columns: {list(df.columns)}')
need = {'signals': ['tic_id', 'period_days', 'epoch_bjd', 'duration_hours', 'depth_ppm', 'signal_label'],
        'lightcurves': ['tic_id', 'hf_path']}
for name, cols in need.items():
    miss = [c for c in cols if c not in (signals if name == 'signals' else lcs).columns]
    assert not miss, f'{name}.csv is missing {miss} - send this error to Claude'
print()
print('signal_label      :', signals['signal_label'].value_counts(dropna=False).to_dict())
if 'tfopwg_disposition' in signals:
    print('tfopwg_disposition:', signals['tfopwg_disposition'].value_counts(dropna=False).to_dict())
if 'file_type' in lcs:
    print('file types        :', lcs['file_type'].value_counts().to_dict())
lab = signals.apply(B.label3_of, axis=1).map({0: 'planet', 1: 'candidate', 2: 'false_positive', -1: 'unknown (dropped)'})
print('our 3-class labels (before de-duplication):', lab.value_counts().to_dict())

## 2 · Plan: which stars fit in 15 GB, with balanced classes

In [ ]:
jobs, summary = B.plan_pull(signals, lcs, BUDGET_GB, FILES_PER_STAR, CAND_RATIO)
print(json.dumps(summary, indent=1))
json.dump(jobs, open(f'{OUT}/jobs.json', 'w'))
for name, df in (('signals.csv', signals), ('lightcurves.csv', lcs), ('targets.csv', targets)):
    df.to_csv(f'{OUT}/{name}', index=False)
cnt = collections.Counter((j['split'], B.CLASSES3[s['label']]) for j in jobs for s in j['signals'])
tab = pd.DataFrame({sp: {c: cnt[(sp, c)] for c in B.CLASSES3} for sp in ('train', 'val', 'test')})
display(tab)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
tab.T.plot.bar(ax=ax[0], color=['C2', 'C7', 'C3']); ax[0].set_title('signals per class and split (planned)')
ax[1].hist([len(j['files']) for j in jobs], bins=np.arange(.5, FILES_PER_STAR + 1.5), color='C1')
ax[1].set_title('sectors per star'); plt.tight_layout(); plt.savefig(f'{OUT}/pull_plan.png'); plt.show()

## 3 · Download → compact .npz → tar (resumable)

In [ ]:
prog = B.pull_compact(OUT, jobs, BUDGET_GB, THREADS, time_budget_h=TIME_BUDGET_H, token=TOKEN,
                      resume_glob='/kaggle/input/**/stars_*.tar')

## 4 · Check what was pulled

In [ ]:
pulled = {t for v in prog.values() for t in v['stars']}
jp = [j for j in jobs if j['tic'] in pulled]
json.dump(jp, open(f'{OUT}/jobs_pulled.json', 'w'))
raw = sum(v['raw_bytes'] for v in prog.values()) / 1e9
cmp = sum(v['tar_bytes'] for v in prog.values()) / 1e9
cnt = collections.Counter(B.CLASSES3[s['label']] for j in jp for s in j['signals'])
print(f'{len(jp)} stars | raw FITS pulled {raw:.2f} GB -> stored {cmp:.2f} GB ({raw / max(cmp, 1e-9):.1f}x smaller)')
print('signals per class:', dict(cnt))
print('stars per split  :', dict(collections.Counter(j['split'] for j in jp)))

# sanity plot: one star of each class, straight from the compact files
import tarfile, tempfile
from astrodetect.pipeline import load_star
tmp = tempfile.mkdtemp()
fig, ax = plt.subplots(3, 1, figsize=(14, 7))
for a, c in zip(ax, range(3)):
    j = next((j for j in jp if j['signals'][0]['label'] == c), None)
    if j is None: continue
    tar = next(k for k, v in prog.items() if j['tic'] in v['stars'])
    with tarfile.open(f'{OUT}/{tar}') as t: t.extract(f"TIC_{j['tic']}.npz", tmp)
    lc = load_star([f"{tmp}/TIC_{j['tic']}.npz"])
    a.plot(lc.time, lc.sap, ',', color='k')
    a.set_title(f"{B.CLASSES3[c]}: TIC {j['tic']} | sectors {lc.meta['sectors']} | P = {j['signals'][0]['period']:.3f} d")
plt.tight_layout(); plt.savefig(f'{OUT}/pull_examples.png'); plt.show()

**Next:** open notebook `01_build_train_3class`, add **this notebook's output** as an input (Add Input → Your Work → Notebooks).
If Kaggle refuses to attach it, use this run's Output tab → *New Dataset*, and attach the dataset instead. Either works.